In [2]:
import pandas as pd

# 1. Import Data and Parse Dates
orders = pd.read_csv("Orders.csv", parse_dates=[
    "Order_Date", "Ship_Date", "Estimated_Delivery_Date", "Actual_Delivery_Date"
])
returns = pd.read_csv("Returns.csv", parse_dates=["Return_Date"])

# 2. Merge Datasets
# Left join so non-returned orders are kept with nulls in return fields
df = orders.merge(returns, on="Order_ID", how="left")

# 3. Engineer New Features
df["Return_Status"] = df["Return_Reason"].notna().map({True: "Yes", False: "No"})
df["Return_Processing_Time"] = (df["Return_Date"] - df["Actual_Delivery_Date"]).dt.days
df["Delivery_Delay_Days"] = (df["Actual_Delivery_Date"] - df["Estimated_Delivery_Date"]).dt.days
df["Was_Late"] = (df["Delivery_Delay_Days"] > 0).astype(int) 

# 4. Clean Financial Columns
# Strip $ and commas before casting to float
for col in ["Sales", "Profit", "Shipping_Cost", "Refund_Amount"]:
    if df[col].dtype == object:
        df[col] = df[col].astype(str).str.replace(r"[$,]", "", regex=True).astype(float)
    df[col] = df[col].astype(float)

# 5. Standardize Data Types and Handle Missing Values
df["Discount"] = df["Discount"].astype(float)
df["Quantity"] = df["Quantity"].astype(int)

# Convert Pandas NaT/NaN to None so MySQL gets proper NULLs instead of errors
df = df.where(pd.notnull(df), None)

# 6. Export the Cleaned Dataset
out_path = "orders_returns_clean.csv"
df.to_csv(out_path, index=False)
print(f"Clean table exported: {out_path} ({len(df)} rows, {df.shape[1]} cols)")

Clean table exported: orders_returns_clean.csv (2000 rows, 23 cols)


In [3]:
df.head


<bound method NDFrame.head of        Order_ID Customer_ID Order_Date  Ship_Date Estimated_Delivery_Date  \
0     ORD-10000   CUST-0000 2023-09-20 2023-09-27              2023-10-01   
1     ORD-10001   CUST-0001 2024-05-20 2024-05-27              2024-05-30   
2     ORD-10002   CUST-0002 2023-09-14 2023-09-17              2023-09-21   
3     ORD-10003   CUST-0003 2024-01-30 2024-02-01              2024-02-04   
4     ORD-10004   CUST-0004 2023-08-07 2023-08-10              2023-08-13   
...         ...         ...        ...        ...                     ...   
1995  ORD-11995   CUST-0395 2024-05-14 2024-05-15              2024-05-18   
1996  ORD-11996   CUST-0396 2024-09-23 2024-09-28              2024-10-02   
1997  ORD-11997   CUST-0397 2023-05-17 2023-05-21              2023-05-25   
1998  ORD-11998   CUST-0398 2023-11-15 2023-11-17              2023-11-20   
1999  ORD-11999   CUST-0399 2023-02-02 2023-02-05              2023-02-08   

     Actual_Delivery_Date     Ship_Mode   Reg

In [4]:
df.describe()


,Order_Date,Ship_Date,Estimated_Delivery_Date,Actual_Delivery_Date,Sales,Quantity,Discount,Profit,Shipping_Cost,Return_Date,Refund_Amount,Return_Processing_Time,Delivery_Delay_Days,Was_Late
count,2000,2000,2000,2000,2000.000000,2000.0000,2000.000000,2000.000000,2000.000000,293,293.000000,293.000000,2000.000000,2000.000000
mean,2023-12-25 16:15:35.999999744,2023-12-28 21:23:02.400000,2023-12-31 21:41:02.400000,2024-01-01 08:34:04.800000,606.166495,4.0370,0.120500,61.153530,30.125905,2024-01-20 03:55:54.266211584,529.245802,6.740614,0.453500,0.251500
min,2023-01-01 00:00:00,2023-01-03 00:00:00,2023-01-05 00:00:00,2023-01-05 00:00:00,15.290000,1.0000,0.000000,-178.160000,2.050000,2023-01-16 00:00:00,12.780000,1.000000,0.000000,0.000000
25%,2023-07-04 00:00:00,2023-07-08 00:00:00,2023-07-10 18:00:00,2023-07-10 18:00:00,301.347500,2.0000,0.000000,-7.782500,15.975000,2023-08-09 00:00:00,267.550000,4.000000,0.000000,0.000000
50%,2023-12-29 00:00:00,2024-01-01 00:00:00,2024-01-04 12:00:00,2024-01-05 00:00:00,610.475000,4.0000,0.100000,31.950000,29.205000,2024-01-24 00:00:00,552.580000,7.000000,0.000000,0.000000
75%,2024-06-19 00:00:00,2024-06-22 06:00:00,2024-06-25 00:00:00,2024-06-25 06:00:00,909.720000,6.0000,0.200000,123.765000,43.922500,2024-07-06 00:00:00,767.760000,10.000000,1.000000,1.000000
max,2024-11-30 00:00:00,2024-12-07 00:00:00,2024-12-09 00:00:00,2024-12-09 00:00:00,1199.310000,7.0000,0.300000,400.980000,60.000000,2024-12-05 00:00:00,1158.760000,13.000000,3.000000,1.000000
std,NaN,NaN,NaN,NaN,348.755345,1.9954,0.116304,108.568346,16.482841,NaN,302.601047,3.585686,0.873624,0.433984


In [7]:
df.duplicated().sum()

np.int64(0)

In [ ]:
df.isnull().sum()[df.isnull().sum() > 0]()